# Kalenjin-English Machine Translation: Zero-Shot Evaluation

## Research Objective
Evaluate the feasibility of translating Kalenjin text to English using pre-trained multilingual translation models without any Kalenjin-specific fine-tuning (zero-shot transfer).

## Methodology
We leverage Meta's NLLB-200 (No Language Left Behind), a multilingual machine translation model covering 200 languages. Since Kalenjin (`kln`) is **not** among the 202 supported language codes, we exploit typological proximity by using related language codes as proxies:

- **Nuer (`nus_Latn`)**: Western Nilotic — closest typological relative in NLLB
- **Luo (`luo_Latn`)**: Western Nilotic — spoken in neighboring regions of Kenya
- **Swahili (`swh_Latn`)**: Bantu — Kenya's national language, different family but shared contact features

## Mathematical Framework

### Sequence-to-Sequence Translation
Given a source sentence $\mathbf{x} = (x_1, x_2, \ldots, x_n)$ in Kalenjin, the translation model estimates:

$$P(\mathbf{y} | \mathbf{x}) = \prod_{t=1}^{m} P(y_t | y_{<t}, \mathbf{x})$$

where $\mathbf{y} = (y_1, y_2, \ldots, y_m)$ is the target English sentence.

### BLEU Score
Translation quality is measured using BLEU (Bilingual Evaluation Understudy):

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

where $p_n$ is the modified n-gram precision, $w_n = 1/N$ are uniform weights, and $BP$ is the brevity penalty:

$$BP = \begin{cases} 1 & \text{if } c > r \\ e^{1 - r/c} & \text{if } c \leq r \end{cases}$$

with $c$ = candidate length and $r$ = reference length.

### chrF Score
We also report chrF (character F-score), which is more robust for morphologically rich languages:

$$\text{chrF}_\beta = (1 + \beta^2) \cdot \frac{\text{chrP} \cdot \text{chrR}}{\beta^2 \cdot \text{chrP} + \text{chrR}}$$

where chrP and chrR are character n-gram precision and recall.

## Pipeline
```
Kalenjin Audio → [ASR: RareElf/kalenjin-asr] → Kalenjin Text → [MT: NLLB-200] → English Text
```

**Model**: `RareElf/kalenjin-asr` (ASR) + `facebook/nllb-200-distilled-600M` (MT)

**Data**: `data/kln/train.tsv`, `data/kln/test.tsv`

## 1. Setup & Dependencies

In [1]:
import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM,
    AutoProcessor, 
    AutoModelForCTC
)
import librosa
from collections import Counter

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

PyTorch: 2.11.0+cpu
Device: cpu


## 2. Load NLLB-200 Translation Model

In [2]:
MT_MODEL = "facebook/nllb-200-distilled-600M"

print(f"Loading translation model: {MT_MODEL}")
mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL).to(DEVICE)
mt_model.eval()

print(f"\u2713 Translation model loaded")
print(f"  Parameters: {sum(p.numel() for p in mt_model.parameters()):,}")

Loading translation model: facebook/nllb-200-distilled-600M


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 30640.26it/s]


✓ Translation model loaded
  Parameters: 615,073,792


## 3. Verify Supported Languages

NLLB-200 uses BCP-47-like language codes with script suffixes (e.g., `eng_Latn`, `swh_Latn`). We verify which Nilotic and East African languages are supported.

In [3]:
# Extract all NLLB language codes
vocab = mt_tokenizer.get_vocab()
lang_codes = sorted([k for k in vocab.keys() if re.match(r'^[a-z]{2,3}_[A-Z][a-z]{3}$', k)])
print(f"Total NLLB language codes: {len(lang_codes)}")

# Check Kalenjin and related languages
print("\n=== Kalenjin (kln) ===")
kln_matches = [l for l in lang_codes if 'kln' in l]
print(f"  Kalenjin: {'FOUND -> ' + str(kln_matches) if kln_matches else 'NOT SUPPORTED'}")

print("\n=== Related Nilotic & East African Languages ===")
related = {
    'nus_Latn': 'Nuer (Western Nilotic — closest relative)',
    'luo_Latn': 'Luo/Dholuo (Western Nilotic — Kenya)',
    'dik_Latn': 'Dinka (Western Nilotic — South Sudan)',
    'swh_Latn': 'Swahili (Bantu — Kenya national language)',
    'kam_Latn': 'Kamba (Bantu — Kenya)',
    'kik_Latn': 'Kikuyu (Bantu — Kenya)',
    'lug_Latn': 'Luganda (Bantu — Uganda)',
    'som_Latn': 'Somali (Cushitic — Horn of Africa)',
}
for code, desc in related.items():
    status = '\u2713 SUPPORTED' if code in lang_codes else '\u2717 NOT FOUND'
    print(f"  {code}: {desc} — {status}")

Total NLLB language codes: 202

=== Kalenjin (kln) ===
  Kalenjin: NOT SUPPORTED

=== Related Nilotic & East African Languages ===
  nus_Latn: Nuer (Western Nilotic — closest relative) — ✓ SUPPORTED
  luo_Latn: Luo/Dholuo (Western Nilotic — Kenya) — ✓ SUPPORTED
  dik_Latn: Dinka (Western Nilotic — South Sudan) — ✓ SUPPORTED
  swh_Latn: Swahili (Bantu — Kenya national language) — ✓ SUPPORTED
  kam_Latn: Kamba (Bantu — Kenya) — ✓ SUPPORTED
  kik_Latn: Kikuyu (Bantu — Kenya) — ✓ SUPPORTED
  lug_Latn: Luganda (Bantu — Uganda) — ✓ SUPPORTED
  som_Latn: Somali (Cushitic — Horn of Africa) — ✓ SUPPORTED


## 4. Load Kalenjin Data

In [4]:
DATA_DIR = Path('../../data/kln')

# Load train and test
train_df = pd.read_csv(DATA_DIR / 'train.tsv', sep='\t')
test_df = pd.read_csv(DATA_DIR / 'test.tsv', sep='\t')

print(f"Train sentences: {len(train_df)}")
print(f"Test sentences: {len(test_df)}")

# Extract unique sentences
train_sentences = train_df['sentence'].dropna().unique().tolist()
test_sentences = test_df['sentence'].dropna().unique().tolist()

print(f"\nUnique train sentences: {len(train_sentences)}")
print(f"Unique test sentences: {len(test_sentences)}")

# Show samples
print("\nSample sentences:")
for i, s in enumerate(test_sentences[:5]):
    print(f"  {i+1}. {s}")

Train sentences: 11065
Test sentences: 5685

Unique train sentences: 11065
Unique test sentences: 5685

Sample sentences:
  1. Kamet tinyei lakwengung kenyisiek ata
  2. Komangen ale tos rikchi chi ko u no
  3. uiy ke nem atepto ne yaa ne kakichop
  4. Kiiyan kochengei logoiwek eng oldo age
  5. Tos iboe nguruonikuk?


## 5. Define Translation Function

In [5]:
def translate(text, src_lang, tgt_lang='eng_Latn', max_length=128):
    """
    Translate text using NLLB-200.
    
    Args:
        text: Source text string
        src_lang: NLLB source language code (e.g., 'nus_Latn')
        tgt_lang: NLLB target language code (default: 'eng_Latn')
        max_length: Maximum output length
    
    Returns:
        Translated text string
    """
    mt_tokenizer.src_lang = src_lang
    inputs = mt_tokenizer(text, return_tensors='pt', max_length=max_length, truncation=True).to(DEVICE)
    
    with torch.no_grad():
        translated = mt_model.generate(
            **inputs,
            forced_bos_token_id=mt_tokenizer.convert_tokens_to_ids(tgt_lang),
            max_new_tokens=max_length,
            num_beams=5,
            early_stopping=True
        )
    
    return mt_tokenizer.batch_decode(translated, skip_special_tokens=True)[0]

# Quick test
test_sent = "Tomo itinye choruet ne chepto iman"
print(f"Input (Kalenjin): {test_sent}")
print(f"Output (via Nuer):    {translate(test_sent, 'nus_Latn')}")
print(f"Output (via Luo):     {translate(test_sent, 'luo_Latn')}")
print(f"Output (via Swahili): {translate(test_sent, 'swh_Latn')}")

Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input (Kalenjin): Tomo itinye choruet ne chepto iman


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output (via Nuer):    I want to add choruet and chepto iman.


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output (via Luo):     I want to add choruet and chepto iman.
Output (via Swahili): I want to add choruet and chepto iman.


## 6. Zero-Shot Translation: Multi-Proxy Evaluation

We translate a sample of Kalenjin sentences using three proxy language codes to determine which produces the most coherent English output. Since we lack English reference translations, we evaluate using:

1. **Output coherence**: Does the English output form grammatically valid sentences?
2. **Semantic plausibility**: Does the output relate to plausible meanings?
3. **Cross-proxy consistency**: Do different proxies produce similar translations?

In [6]:
N_SAMPLES = 30
proxy_langs = ['nus_Latn', 'luo_Latn', 'swh_Latn']
proxy_names = {'nus_Latn': 'Nuer', 'luo_Latn': 'Luo', 'swh_Latn': 'Swahili'}

# Select diverse samples (short, medium, long)
sample_sentences = test_sentences[:N_SAMPLES]

results = []
for i, sent in enumerate(sample_sentences):
    row = {'idx': i, 'kalenjin': sent}
    for lang in proxy_langs:
        try:
            translation = translate(sent, lang)
            row[f'{proxy_names[lang]}_translation'] = translation
        except Exception as e:
            row[f'{proxy_names[lang]}_translation'] = f'ERROR: {e}'
    results.append(row)
    
    if i < 10:
        print(f"\n[{i+1}] KLN: {sent}")
        for lang in proxy_langs:
            name = proxy_names[lang]
            print(f"     {name:>7}: {row[f'{name}_translation']}")

print(f"\n\u2713 Translated {len(results)} sentences with {len(proxy_langs)} proxy languages")

Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[1] KLN: Kamet tinyei lakwengung kenyisiek ata
        Nuer: It's time to put on a leather jacket.
         Luo: It's time to put on a leather jacket.
     Swahili: It's time to put on a leather jacket.


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[2] KLN: Komangen ale tos rikchi chi ko u no
        Nuer: Comangen ale to rikchi chi ko u no
         Luo: Comangen ale to rikchi chi ko u no
     Swahili: Comangen ale to rikchi chi ko u no


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


[9] KLN: mi barak olyetab somanet ago nyoru bik chechang che ngering neya
        Nuer: Barak mi olyetab somanet ago nyoru bik chechang that ngering neya
         Luo: Barak mi olyetab somanet ago nyoru bik chechang that ngering neya
     Swahili: Barak mi olyetab somanet ago nyoru bik chechang that ngering neya


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[10] KLN: Nito ko karit tab maat nechoktoi mising
        Nuer: Nito ko karit tab maat nechoktoi is missing
         Luo: Nito ko karit tab maat nechoktoi is missing
     Swahili: Nito ko karit tab maat nechoktoi is missing


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✓ Translated 30 sentences with 3 proxy languages


## 7. Translation Quality Analysis

In [7]:
# Analyze output characteristics
for lang in proxy_langs:
    name = proxy_names[lang]
    translations = [r[f'{name}_translation'] for r in results]
    
    # Basic stats
    avg_len = np.mean([len(t.split()) for t in translations])
    empty = sum(1 for t in translations if not t.strip() or t.startswith('ERROR'))
    
    # Check if output is just copying input (sign of failure)
    copies = sum(1 for r in results if r[f'{name}_translation'].lower().strip() == r['kalenjin'].lower().strip())
    
    # Check English word ratio (rough measure of actual translation)
    common_english = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'of', 'in', 'to', 'and', 'for', 'that', 'it', 'he', 'she', 'they', 'we', 'you', 'not', 'with', 'this', 'but', 'from', 'have', 'has', 'had', 'will', 'would', 'can', 'could', 'do', 'does', 'did', 'be', 'been', 'being'}
    eng_word_counts = []
    for t in translations:
        words = t.lower().split()
        eng_count = sum(1 for w in words if w in common_english)
        eng_word_counts.append(eng_count / max(len(words), 1))
    avg_eng_ratio = np.mean(eng_word_counts)
    
    print(f"\n=== {name} ({lang}) ===")
    print(f"  Avg output length: {avg_len:.1f} words")
    print(f"  Empty/error outputs: {empty}/{len(results)}")
    print(f"  Direct copies (no translation): {copies}/{len(results)}")
    print(f"  English word ratio: {avg_eng_ratio:.2%}")


=== Nuer (nus_Latn) ===
  Avg output length: 8.3 words
  Empty/error outputs: 0/30
  Direct copies (no translation): 3/30
  English word ratio: 15.64%

=== Luo (luo_Latn) ===
  Avg output length: 8.3 words
  Empty/error outputs: 0/30
  Direct copies (no translation): 3/30
  English word ratio: 15.64%

=== Swahili (swh_Latn) ===
  Avg output length: 8.3 words
  Empty/error outputs: 0/30
  Direct copies (no translation): 3/30
  English word ratio: 15.64%


## 8. Cross-Proxy Consistency Analysis

We measure how consistent the translations are across different proxy languages. High consistency suggests the model is capturing actual meaning rather than producing random output.

In [8]:
from difflib import SequenceMatcher

def similarity(a, b):
    """Compute string similarity ratio between two texts."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

# Compare each pair of proxy translations
pairs = [('Nuer', 'Luo'), ('Nuer', 'Swahili'), ('Luo', 'Swahili')]

print("=== Cross-Proxy Translation Similarity ===")
for name_a, name_b in pairs:
    sims = []
    for r in results:
        a = r[f'{name_a}_translation']
        b = r[f'{name_b}_translation']
        sims.append(similarity(a, b))
    print(f"  {name_a} vs {name_b}: mean={np.mean(sims):.3f}, median={np.median(sims):.3f}")

print("\n(Higher similarity = more consistent translations across proxies)")

=== Cross-Proxy Translation Similarity ===
  Nuer vs Luo: mean=1.000, median=1.000
  Nuer vs Swahili: mean=1.000, median=1.000
  Luo vs Swahili: mean=1.000, median=1.000

(Higher similarity = more consistent translations across proxies)


## 9. End-to-End Pipeline: Kalenjin Audio → English Text

Combine the ASR model with the best-performing translation proxy to create a complete speech translation pipeline.

In [ ]:
# Load ASR model
ASR_MODEL = "RareElf/kalenjin-asr"
print(f"Loading ASR model: {ASR_MODEL}")
asr_processor = AutoProcessor.from_pretrained(ASR_MODEL)
asr_model = AutoModelForCTC.from_pretrained(ASR_MODEL).to(DEVICE)
asr_model.eval()
print(f"\u2713 ASR model loaded")

In [ ]:
def speech_to_english(audio_path, proxy_lang='luo_Latn'):
    """
    End-to-end: Kalenjin audio → Kalenjin text → English text.
    
    Pipeline:
        1. Load and resample audio to 16kHz
        2. ASR: Wav2Vec2-XLS-R → Kalenjin text (greedy decoding)
        3. MT: NLLB-200 → English text (beam search, proxy language)
    
    Args:
        audio_path: Path to audio file (MP3/WAV)
        proxy_lang: NLLB language code to use as source proxy
    
    Returns:
        dict with kalenjin_text and english_text
    """
    # Step 1: Load audio
    audio, sr = librosa.load(audio_path, sr=16000)
    
    # Step 2: ASR
    inputs = asr_processor(audio, sampling_rate=16000, return_tensors='pt', padding=True).to(DEVICE)
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)
    kalenjin_text = asr_processor.batch_decode(pred_ids)[0].strip()
    
    # Step 3: Translation
    english_text = translate(kalenjin_text, proxy_lang) if kalenjin_text else ''
    
    return {
        'audio_path': str(audio_path),
        'kalenjin_text': kalenjin_text,
        'english_text': english_text,
        'proxy_lang': proxy_lang
    }

print("\u2713 End-to-end pipeline defined")

## 10. Test End-to-End Pipeline on Audio Samples

In [ ]:
CLIPS_DIR = DATA_DIR / 'clips'

# Get test audio files
test_df['audio_path'] = test_df['path'].apply(lambda p: CLIPS_DIR / p)
test_df['exists'] = test_df['audio_path'].apply(lambda p: p.exists())
valid_test = test_df[test_df['exists']].reset_index(drop=True)

print(f"Test samples with audio: {len(valid_test)}")

# Run pipeline on 20 samples
N_E2E = 20
BEST_PROXY = 'luo_Latn'  # Update based on Section 7 results

e2e_results = []
for i in range(min(N_E2E, len(valid_test))):
    row = valid_test.iloc[i]
    try:
        result = speech_to_english(str(row['audio_path']), proxy_lang=BEST_PROXY)
        result['reference_kalenjin'] = row['sentence']
        e2e_results.append(result)
        
        print(f"\n[{i+1}] Audio: {row['path']}")
        print(f"     Reference (KLN): {row['sentence']}")
        print(f"     ASR output (KLN): {result['kalenjin_text']}")
        print(f"     Translation (ENG): {result['english_text']}")
    except Exception as e:
        print(f"[{i+1}] Error: {e}")

print(f"\n\u2713 Processed {len(e2e_results)} end-to-end samples")

## 11. Save Results

In [ ]:
# Save zero-shot translation results
output = {
    'mt_model': MT_MODEL,
    'asr_model': ASR_MODEL,
    'proxy_languages': list(proxy_names.keys()),
    'n_translation_samples': len(results),
    'n_e2e_samples': len(e2e_results),
    'translation_results': results,
    'e2e_results': e2e_results
}

output_path = Path('../evaluation/zero_shot_results.json')
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(output, f, indent=2, default=str)

print(f"Results saved to {output_path}")

## 12. Summary & Next Steps

### Findings
- Kalenjin (`kln`) is **not** directly supported by NLLB-200
- Zero-shot translation via proxy languages produces [evaluate based on results above]
- Best proxy language: [determine from Section 7]

### Next Steps
1. **Collect parallel data**: Kalenjin-English sentence pairs from Bible translations, JW300, educational materials
2. **Fine-tune NLLB-200**: Adapt the model to Kalenjin using parallel data
3. **Evaluate with BLEU/chrF**: Once we have reference translations
4. **Build Gradio demo**: Interactive speech-to-translation interface
5. **Publish model**: Push fine-tuned translation model to HuggingFace